<a href="https://colab.research.google.com/github/EmanSalah2000/Langchain_projects/blob/main/TextToSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#text to sql using Langchain

# **Install**

In [2]:
!pip install -qU langchain langchain-openai langchain-community gradio sqlparse pandas  openai
print("✅ Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB

# **DataBase Setup&Read&connect**

In [3]:
# ── Cell 2: Download the Chinook database ─────────────────────────────────────
import os

DB_PATH = "/content/Chinook_Sqlite.sqlite"
DB_URL  = "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"

if not os.path.exists(DB_PATH):
    !wget -q --show-progress -O {DB_PATH} {DB_URL}
    print(f"✅ Downloaded → {DB_PATH}")
else:
    print(f"✅ Already exists → {DB_PATH}")

/content/Chinook_Sq 100%[===================>]   1.02M  --.-KB/s    in 0.06s   
✅ Downloaded → /content/Chinook_Sqlite.sqlite


In [4]:
# Print only the name of each table in DB
#import sqlite3

# # connect to database file
# conn = sqlite3.connect("/content/Chinook_Sqlite.sqlite")

# cursor = conn.cursor()

# # check tables
# cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
# print(cursor.fetchall())
# conn.close()

In [5]:
#Print the columns in each table in DB
import sqlite3

def get_schema_context(db_path: str) -> str:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    tables = [row[0] for row in cursor.fetchall()]

    schema_parts = []

    for table in tables:
        # Get column info
        cursor.execute(f"PRAGMA table_info({table});")
        columns = cursor.fetchall()

        # Get foreign keys
        cursor.execute(f"PRAGMA foreign_key_list({table});")
        foreign_keys = cursor.fetchall()

        # Build column descriptions
        col_lines = []
        for col in columns:
            col_id, col_name, col_type, not_null, default, is_pk = col
            pk_tag = " [PK]" if is_pk else ""
            col_lines.append(f"    - {col_name} ({col_type}){pk_tag}")

        # Build foreign key descriptions
        fk_lines = []
        for fk in foreign_keys:
            fk_lines.append(f"    - {fk[3]} → {fk[2]}.{fk[4]}")

        # Assemble table block
        block = f"Table: {table}\n  Columns:\n" + "\n".join(col_lines)
        if fk_lines:
            block += "\n  Foreign Keys:\n" + "\n".join(fk_lines)

        schema_parts.append(block)

    conn.close()
    return "\n\n".join(schema_parts)


SCHEMA_CONTEXT = get_schema_context(DB_PATH)
# print(SCHEMA_CONTEXT) # ALL Shema of each table and each columns
# Preview just the first table to confirm it looks right
preview = SCHEMA_CONTEXT.split("\n\n")[0]
print("── Preview (first table) ──")
print(preview)

── Preview (first table) ──
Table: Album
  Columns:
    - AlbumId (INTEGER) [PK]
    - Title (NVARCHAR(160))
    - ArtistId (INTEGER)
  Foreign Keys:
    - ArtistId → Artist.ArtistId


# **Inverst Prompt**

In [6]:
from langchain_core.prompts import ChatPromptTemplate

In [7]:
SQL_PROMPT_TEMPLATE = """
You are an expert SQL assistant. Your job is to convert natural language questions
into valid SQLite SQL queries based on the database schema provided.

=== DATABASE SCHEMA ===
{schema}

=== RULES ===
1. Generate ONLY a SELECT statement — never INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, or TRUNCATE.
2. Use exact table and column names from the schema above.
3. Always use JOINs when data spans multiple tables.
4. For aggregations, always include a GROUP BY clause.
5. If the question is unanswerable from this schema, reply with: UNANSWERABLE
6. Do NOT include markdown formatting or code blocks in your SQL.

=== FEW-SHOT EXAMPLES ===

Question: How many artists are in the database?
SQL: SELECT COUNT(*) AS TotalArtists FROM Artist;
Explanation: Counts all rows in the Artist table.

Question: Which 10 artists have the most tracks?
SQL: SELECT ar.Name, COUNT(t.TrackId) AS TrackCount FROM Artist ar JOIN Album al ON ar.ArtistId = al.ArtistId JOIN Track t ON al.AlbumId = t.AlbumId GROUP BY ar.ArtistId ORDER BY TrackCount DESC LIMIT 10;
Explanation: Joins Artist → Album → Track, groups by artist, orders by track count descending.

Question: What are the top 5 best-selling tracks by quantity sold?
SQL: SELECT t.Name, SUM(il.Quantity) AS TotalSold FROM Track t JOIN InvoiceLine il ON t.TrackId = il.TrackId GROUP BY t.TrackId ORDER BY TotalSold DESC LIMIT 5;
Explanation: Joins Track with InvoiceLine to sum quantities sold per track.

=== YOUR TASK ===
Question: {question}

Respond in this exact format:
SQL: <your SQL query here>
Explanation: <one sentence explaining what the query does>
"""

prompt = ChatPromptTemplate.from_template(SQL_PROMPT_TEMPLATE)
print("✅ Prompt template created")

✅ Prompt template created


In [8]:
questions = {
    "Simple": [
        "How many artists are in the database?",
        "List all genre names.",
        "What is the most expensive track?",
        "How many customers are there?",
        "Show the 5 longest tracks by duration in milliseconds."
    ],

    "Intermediate": [
        "Which albums were released by AC/DC?",
        "How many tracks does each genre have? Order by count descending.",
        "List all customers from Brazil.",
        "What are the top 5 best-selling tracks by total quantity sold?",
        "Which employee supports the most customers?",
        "How much total revenue was generated per country?",
        "What are the top 10 artists by number of albums?"
    ],

    "Advanced": [
        "Which 10 artists have generated the most total revenue?",
        "What is the average invoice total per customer?",
        "Which customers have spent more than the average customer?",
        "What are the top 5 playlists by number of tracks?",
        "How many invoices were placed each year? Order by year.",
        "Which genres generate the most total revenue?"
    ],

    "Edge case": [
        "Show all Rock tracks that cost more than $1.00.",
        "Which customers have never placed an order?"
    ]
}

In [9]:
# sample_messages = prompt.format_messages(
#     schema=SCHEMA_CONTEXT,
#     question=questions["Simple"][0]
# )
# print(✅ Prompts Are Ready)

# **OpenAI SetUP**

In [12]:
#openAI KEY
import dotenv

_=dotenv.load_dotenv("/content/env.env") # Write Your OpenAI key HERE
print("✅ API key set")

✅ API key set


# **LLM Chain**

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# LLM — gpt-4o-mini is fast and cheap, swap to gpt-4o for better SQL accuracy
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,        # 0 = deterministic, important for SQL generation
    max_tokens=500
)

# Output parser — just extracts the raw text from the LLM response
output_parser = StrOutputParser()

# The LCEL chain: prompt | llm | output_parser
sql_chain = prompt | llm | output_parser

print("✅ LCEL chain ready: prompt | llm | output_parser")

✅ LCEL chain ready: prompt | llm | output_parser


# **Paser Output for SQL Explaniation**

In [14]:
def parse_response(raw: str) -> dict:
    """
    Splits the LLM output into SQL and Explanation.
    Expected format:
        SQL: SELECT ...
        Explanation: ...
    """
    sql, explanation = "", ""

    for line in raw.strip().splitlines():
        if line.startswith("SQL:"):
            sql = line[len("SQL:"):].strip()
        elif line.startswith("Explanation:"):
            explanation = line[len("Explanation:"):].strip()

    # Handle multi-line SQL (if model wraps across lines)
    if not sql:
        sql = raw.strip()

    return {"sql": sql, "explanation": explanation, "raw": raw}


print("✅ Response parser defined")

✅ Response parser defined


In [15]:
def run_query(question: str) -> dict:
    """
    Full pipeline:
    question → prompt → LLM → parse → return sql + explanation
    """
    print(f"\n🔍 Question: {question}")

    # Step 1: call the chain
    raw_response = sql_chain.invoke({
        "schema": SCHEMA_CONTEXT,
        "question": question
    })

    # Step 2: parse SQL and explanation out of the response
    parsed = parse_response(raw_response)

    print(f"📝 SQL:         {parsed['sql']}")
    print(f"💬 Explanation: {parsed['explanation']}")

    return parsed


print("✅ run_query() function ready")

✅ run_query() function ready


In [16]:
results_simple = []
for q in questions["Simple"]:
    result = run_query(q)
    results_simple.append(result)
    print("-" * 60)

results_intermediate = []
for q in questions["Intermediate"]:
    result = run_query(q)
    results_intermediate.append(result)
    print("-" * 60)

results_advanced = []
for q in questions["Advanced"]:
    result = run_query(q)
    results_advanced.append(result)
    print("-" * 60)

result_edge = []
for q in questions["Edge case"]:
    result = run_query(q)
    result_edge.append(result)
    print("-" * 60)


🔍 Question: How many artists are in the database?
📝 SQL:         SELECT COUNT(*) AS TotalArtists FROM Artist;
💬 Explanation: Counts all rows in the Artist table.
------------------------------------------------------------

🔍 Question: List all genre names.
📝 SQL:         SELECT Name FROM Genre;
💬 Explanation: Selects all genre names from the Genre table.
------------------------------------------------------------

🔍 Question: What is the most expensive track?
📝 SQL:         SELECT t.Name, t.UnitPrice FROM Track t ORDER BY t.UnitPrice DESC LIMIT 1;
💬 Explanation: This query selects the name and unit price of the track with the highest price by ordering the tracks in descending order of their unit price and limiting the result to one.
------------------------------------------------------------

🔍 Question: How many customers are there?
📝 SQL:         SELECT COUNT(*) AS TotalCustomers FROM Customer;
💬 Explanation: Counts all rows in the Customer table.
--------------------------------

# **Safety module**
IF LLM Halucinate and return a blocked list [INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, TRUNCATE] and multi-statement queries (anything with ; in the middle)

In [17]:

import sqlparse
from sqlparse.sql import Statement
from sqlparse.tokens import Keyword, DDL, DML

print("✅ sqlparse ready")

✅ sqlparse ready


In [18]:
BLOCKED_KEYWORDS = {
    "INSERT", "UPDATE", "DELETE", "DROP",
    "ALTER", "CREATE", "TRUNCATE", "REPLACE",
    "ATTACH", "DETACH", "PRAGMA"
}

In [19]:
def is_safe_sql(sql: str) -> tuple[bool, str]:
    """
    Validates that the SQL is a pure read-only SELECT statement.
    Returns (is_safe: bool, reason: str)
    """
    if not sql or not sql.strip():
        return False, "Empty query received."

    # ── Check 1: block multi-statement queries (e.g. SELECT ...; DROP TABLE)
    # Strip the trailing semicolon first, then check for any remaining ones
    cleaned = sql.strip().rstrip(";")
    if ";" in cleaned:
        return False, "Multi-statement query detected — only single statements are allowed."

    # ── Check 2: parse and inspect token types with sqlparse
    parsed = sqlparse.parse(sql)
    if not parsed:
        return False, "Could not parse the SQL query."

    statement: Statement = parsed[0]

    # ── Check 3: must be a SELECT statement
    statement_type = statement.get_type()
    if statement_type != "SELECT":
        return False, f"Only SELECT queries are allowed. Detected type: '{statement_type}'."

    # ── Check 4: scan every token for blocked keywords (extra safety net)
    for token in statement.flatten():
        token_upper = token.value.upper()
        if token_upper in BLOCKED_KEYWORDS:
            return False, f"Blocked keyword detected: '{token.value}'."

    return True, "Query is safe ✅"

In [20]:
import pandas as pd

def execute_safe_query(sql: str) -> tuple[pd.DataFrame | None, str]:
    """
    Runs the SQL only if it passes the safety check.
    Returns (dataframe or None, status message)
    """
    # Safety gate
    safe, reason = is_safe_sql(sql)
    if not safe:
        return None, f"🚫 Blocked: {reason}"

    # Execute
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query(sql, conn)
        conn.close()

        if df.empty:
            return df, "⚠️ Query ran successfully but returned no results."

        return df, f"✅ Success — {len(df)} row(s) returned."

    except Exception as e:
        return None, f"❌ Execution error: {str(e)}"

print("✅ execute_safe_query() function defined")

✅ execute_safe_query() function defined


In [21]:
print("="*50)
print("TEST CASES OUT OF PROJECT")
print("="*50)

test_cases = [
    # (sql, expected)
    ("SELECT * FROM Artist LIMIT 5",                        "✅ PASS"),
    ("SELECT COUNT(*) FROM Track",                          "✅ PASS"),
    ("DROP TABLE Artist",                                   "🚫 BLOCK"),
    ("DELETE FROM Customer WHERE CustomerId = 1",           "🚫 BLOCK"),
    ("SELECT * FROM Artist; DROP TABLE Artist",             "🚫 BLOCK"),
    ("INSERT INTO Artist VALUES (999, 'Hacker')",           "🚫 BLOCK"),
    ("UPDATE Artist SET Name='X' WHERE ArtistId=1",        "🚫 BLOCK"),
]

print("── Safety Module Tests ──\n")
for sql, expected in test_cases:
    safe, reason = is_safe_sql(sql)
    status = "✅ PASS" if safe else "🚫 BLOCK"
    match = "✓" if status == expected else "✗ MISMATCH"
    print(f"{match}  [{status}]  {sql[:55]:<55}  → {reason}")

TEST CASES OUT OF PROJECT
── Safety Module Tests ──

✓  [✅ PASS]  SELECT * FROM Artist LIMIT 5                             → Query is safe ✅
✓  [✅ PASS]  SELECT COUNT(*) FROM Track                               → Query is safe ✅
✓  [🚫 BLOCK]  DROP TABLE Artist                                        → Only SELECT queries are allowed. Detected type: 'DROP'.
✓  [🚫 BLOCK]  DELETE FROM Customer WHERE CustomerId = 1                → Only SELECT queries are allowed. Detected type: 'DELETE'.
✓  [🚫 BLOCK]  SELECT * FROM Artist; DROP TABLE Artist                  → Multi-statement query detected — only single statements are allowed.
✓  [🚫 BLOCK]  INSERT INTO Artist VALUES (999, 'Hacker')                → Only SELECT queries are allowed. Detected type: 'INSERT'.
✓  [🚫 BLOCK]  UPDATE Artist SET Name='X' WHERE ArtistId=1              → Only SELECT queries are allowed. Detected type: 'UPDATE'.


In [22]:
ALL_Results = {
    "Simple": results_simple,
    "Intermediate": results_intermediate,
    "Advanced": results_advanced,
    "Edge case": result_edge
}

for level, results in ALL_Results.items():

    print("\n" + "=" * 70)
    print(f"📌 LEVEL: {level}")
    print("=" * 70)

    for question, result in zip(questions[level], results):

        df, status = execute_safe_query(result['sql'])

        print("\n" + "-" * 60)
        print(f"❓ Question: {question}")
        print(f"📝 SQL: {result['sql']}")
        print(f"🔒 Safety status: {status}")

        if df is not None:
            print("\n📊 Results:")
            print(df.to_string(index=False))

        print("-" * 60)


📌 LEVEL: Simple

------------------------------------------------------------
❓ Question: How many artists are in the database?
📝 SQL: SELECT COUNT(*) AS TotalArtists FROM Artist;
🔒 Safety status: ✅ Success — 1 row(s) returned.

📊 Results:
 TotalArtists
          275
------------------------------------------------------------

------------------------------------------------------------
❓ Question: List all genre names.
📝 SQL: SELECT Name FROM Genre;
🔒 Safety status: ✅ Success — 25 row(s) returned.

📊 Results:
              Name
              Rock
              Jazz
             Metal
Alternative & Punk
     Rock And Roll
             Blues
             Latin
            Reggae
               Pop
        Soundtrack
        Bossa Nova
    Easy Listening
       Heavy Metal
          R&B/Soul
 Electronica/Dance
             World
       Hip Hop/Rap
   Science Fiction
          TV Shows
  Sci Fi & Fantasy
             Drama
            Comedy
       Alternative
         Classical
       

# **Save TO Excel Sheet**

In [25]:
# Expected SQL mapping — fill these in for your questions
expected_sql = {
    "Simple": [
        "SELECT COUNT(*) FROM Artist",
        "SELECT Name FROM Genre",
        "SELECT Name, UnitPrice FROM Track ORDER BY UnitPrice DESC LIMIT 1",
        "SELECT COUNT(*) FROM Customer",
        "SELECT Name, Milliseconds FROM Track ORDER BY Milliseconds DESC LIMIT 5",
    ],
    "Intermediate": [
        "SELECT al.Title FROM Album al JOIN Artist a ON al.ArtistId = a.ArtistId WHERE a.Name = 'AC/DC'",
        "SELECT g.Name, COUNT(t.TrackId) AS TrackCount FROM Genre g JOIN Track t ON g.GenreId = t.GenreId GROUP BY g.GenreId ORDER BY TrackCount DESC",
        "SELECT * FROM Customer WHERE Country = 'Brazil'",
        "SELECT t.Name, SUM(il.Quantity) AS TotalSold FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId GROUP BY il.TrackId ORDER BY TotalSold DESC LIMIT 5",
        "SELECT e.FirstName, e.LastName, COUNT(c.CustomerId) AS CustomerCount FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId GROUP BY e.EmployeeId ORDER BY CustomerCount DESC LIMIT 1",
        "SELECT BillingCountry, SUM(Total) AS Revenue FROM Invoice GROUP BY BillingCountry ORDER BY Revenue DESC",
        "SELECT a.Name, COUNT(al.AlbumId) AS AlbumCount FROM Artist a JOIN Album al ON a.ArtistId = al.ArtistId GROUP BY a.ArtistId ORDER BY AlbumCount DESC LIMIT 10",
    ],
    "Advanced": [
        "SELECT a.Name, SUM(il.UnitPrice * il.Quantity) AS Revenue FROM Artist a JOIN Album al ON a.ArtistId = al.ArtistId JOIN Track t ON al.AlbumId = t.AlbumId JOIN InvoiceLine il ON t.TrackId = il.TrackId GROUP BY a.ArtistId ORDER BY Revenue DESC LIMIT 10",
        "SELECT c.FirstName, c.LastName, AVG(i.Total) AS AvgInvoice FROM Customer c JOIN Invoice i ON c.CustomerId = i.CustomerId GROUP BY c.CustomerId",
        "SELECT c.FirstName, c.LastName, SUM(i.Total) AS Total FROM Customer c JOIN Invoice i ON c.CustomerId = i.CustomerId GROUP BY c.CustomerId HAVING Total > (SELECT AVG(Total) FROM Invoice)",
        "SELECT p.Name, COUNT(pt.TrackId) AS TrackCount FROM Playlist p JOIN PlaylistTrack pt ON p.PlaylistId = pt.PlaylistId GROUP BY p.PlaylistId ORDER BY TrackCount DESC LIMIT 5",
        "SELECT strftime('%Y', InvoiceDate) AS Year, COUNT(*) AS InvoiceCount FROM Invoice GROUP BY Year ORDER BY Year",
        "SELECT g.Name, SUM(il.UnitPrice * il.Quantity) AS Revenue FROM Genre g JOIN Track t ON g.GenreId = t.GenreId JOIN InvoiceLine il ON t.TrackId = il.TrackId GROUP BY g.GenreId ORDER BY Revenue DESC",
    ],
    "Edge case": [
        "SELECT t.Name, t.UnitPrice FROM Track t JOIN Album al ON t.AlbumId = al.AlbumId JOIN Genre g ON t.GenreId = g.GenreId WHERE g.Name = 'Rock' AND t.UnitPrice > 1.00",
        "SELECT c.FirstName, c.LastName FROM Customer c LEFT JOIN Invoice i ON c.CustomerId = i.CustomerId WHERE i.InvoiceId IS NULL",
    ],
}

In [28]:
import csv

# ── collect rows ──────────────────────────────────────────────────────────────
rows = []
for level, results in ALL_Results.items():
    for i, (question, result) in enumerate(zip(questions[level], results)):
        df, status = execute_safe_query(result["sql"])

        if df is not None and not df.empty:
            result_str = df.to_string(index=False)
        elif df is not None:
            result_str = "(no rows returned)"
        else:
            result_str = "(blocked or error)"

        rows.append({
            "Level":         level,
            "Question":      question,
            "Expected SQL":  expected_sql[level][i] if i < len(expected_sql[level]) else "",
            "Generated SQL": result["sql"],
            "Status":        status,
            "Result":        result_str,
        })

        # keep the original console output
        print("\n" + "-" * 60)
        print(f"❓ Question: {question}")
        print(f"📝 SQL: {result['sql']}")
        print(f"🔒 Safety status: {status}")
        if df is not None:
            print("\n📊 Results:")
            print(df.to_string(index=False))
        print("-" * 60)

# ── save CSV ──────────────────────────────────────────────────────────────────
output_path = "evaluation_results.csv"
fieldnames = ["Level", "Question", "Expected SQL", "Generated SQL", "Status", "Result"]

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"\n✅ Saved evaluation sheet → {output_path}")


------------------------------------------------------------
❓ Question: How many artists are in the database?
📝 SQL: SELECT COUNT(*) AS TotalArtists FROM Artist;
🔒 Safety status: ✅ Success — 1 row(s) returned.

📊 Results:
 TotalArtists
          275
------------------------------------------------------------

------------------------------------------------------------
❓ Question: List all genre names.
📝 SQL: SELECT Name FROM Genre;
🔒 Safety status: ✅ Success — 25 row(s) returned.

📊 Results:
              Name
              Rock
              Jazz
             Metal
Alternative & Punk
     Rock And Roll
             Blues
             Latin
            Reggae
               Pop
        Soundtrack
        Bossa Nova
    Easy Listening
       Heavy Metal
          R&B/Soul
 Electronica/Dance
             World
       Hip Hop/Rap
   Science Fiction
          TV Shows
  Sci Fi & Fantasy
             Drama
            Comedy
       Alternative
         Classical
             Opera
-----

# **Gradio UI**

In [24]:
import gradio as gr
import pandas as pd


# ---------------------------------
# Function
# ---------------------------------
def output_function(question):

    result = run_query(question)

    df, status = execute_safe_query(result['sql'])

    # If query succeeded
    if df is not None:

        return (
            question,           # Question
            result['sql'],      # SQL
            status,             # Status
            df                  # DataFrame
        )

    # If query failed
    return (
        question,
        result['sql'],
        status,
        pd.DataFrame()
    )


# ---------------------------------
# Gradio UI
# ---------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🗄️ AI SQL Query Generator
    ### Ask Questions About the Database
    """)

    # Input
    question_input = gr.Textbox(
        label="❓ Enter Your Question",
        placeholder="Example: How many artists are in the database?"
    )

    run_btn = gr.Button(
        "🚀 Run Query",
        variant="primary"
    )

    # Outputs
    question_output = gr.Textbox(
        label="📌 Question"
    )

    sql_output = gr.Code(
        label="📝 Generated SQL",
        language="sql"
    )

    status_output = gr.Textbox(
        label="🔒 Safety Status"
    )

    dataframe_output = gr.DataFrame(
        label="📊 Query Result",
        interactive=False
    )

    # Button Action
    run_btn.click(
        fn=output_function,
        inputs=question_input,
        outputs=[
            question_output,
            sql_output,
            status_output,
            dataframe_output
        ]
    )

# Launch App
demo.launch()

/tmp/ipykernel_5170/3457975532.py:36: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9473ef51c7caef5e3c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
